# Feature Extraction from Pretrained Models

Este notebook permite extrair features de modelos pré-treinados e salvar em pastas customizadas.

In [61]:
import sys
sys.path.insert(0, '/home/eflammere/HybridEmbeddingsForBreastCancerDetection')

import os
import numpy as np
import torch
from tqdm import tqdm

from src.utils.setup import setup_device
from src.utils.dataloader import BreastMNISTLoader

# Importar TODOS os modelos quânticos
from src.models.main.models import (
    QuantumModel,                      # Quantum sem encoder (non-trainable)
    QuantumModelWithEncoder,           # Quantum com encoder (non-trainable)
    ExtentedQuantumModel,              # Quantum sem encoder (trainable, 3 FC layers)
    ExtentedQuantumModelWithEncoder,   # Quantum com encoder (trainable, 3 FC layers)
)

# Importar TODOS os modelos clássicos
from src.models.main.classical import (
    ClassicalModel,                    # ResNet18 sem encoder        
    ShallowCNN,                        # CNN rasa sem encoder      
)

# Importar TODOS os extractors disponíveis
from src.models.extraction import (
    create_extractor,                         # Helper para criar extractor automaticamente
)

## 1. Configurações

Defina o experimento em **um único lugar** (`EXPERIMENT_CONFIG`).
A partir dele, o notebook resolve automaticamente:
- paths dos checkpoints (clássico e quântico)
- pasta de saída
- parâmetros de loader/modelo

In [62]:
# ============================================================================
# CONFIGURAÇÃO ÚNICA DO EXPERIMENTO (edite apenas aqui)
# ============================================================================

EXPERIMENT_CONFIG = {
    "dataset": "INbreast",              # Opções: BreastMNIST, BUSUCLM, INbreast
    "backbone": "shallow",             # Opções: shallow, resnet
    "quantum_variant": "basic",    # Opções: basic, trainable
    "num_classes": 2,
    "image_size": 28,
    "batch_size": 8,
    "n_qubits": 4,
    "n_layers": 1,
    "target_feature_dim": 128,          # None para manter dimensão original
    "output_root": "experiments/features",
}

# ============================================================================
# MAPEAMENTO CENTRALIZADO DE CHECKPOINTS
# ============================================================================

MODEL_REGISTRY = {
    "BreastMNIST": {
        "classical": {
            "resnet": "../experiments/models/classical/BreastMNIST/resnet/best_resnet.pth",
            "shallow": "../experiments/models/classical/BreastMNIST/shallow/best_shallow.pth",
        },
        "quantum": {
            "basic": "../experiments/models/quantum/basic/BreastMNIST/checkpoints/model_epoch16.pth",
            "trainable": "../experiments/models/quantum/trainable/BreastMNIST/checkpoints/model_epoch15.pth",
        },
    },
    "BUSUCLM": {
        "classical": {
            "resnet": "../experiments/models/classical/BUSUCLM/resnet/best_resnet.pth",
            "shallow": "../experiments/models/classical/BUSUCLM/shallow/best_shallow.pth",
        },
        "quantum": {
            "basic": "../experiments/models/quantum/basic/BUSUCLM/checkpoints/model_epoch18.pth",
            "trainable": "../experiments/models/quantum/trainable/BUSUCLM/checkpoints/model_epoch5.pth",
        },
    },
    "INbreast": {
        "classical": {
            "resnet": "../experiments/models/classical/INbreast/resnet/best_resnet.pth",
            "shallow": "../experiments/models/classical/INbreast/shallow/best_shallow.pth",
        },
        "quantum": {
            "basic": "../experiments/models/quantum/basic/INbreast/checkpoints/model_epoch4.pth",
            "trainable": "../experiments/models/quantum/trainable/INbreast/checkpoints/model_epoch20.pth",
        },
    },
}

def _resolve_experiment_paths(config):
    dataset = config["dataset"]
    backbone = config["backbone"]
    quantum_variant = config["quantum_variant"]

    if dataset not in MODEL_REGISTRY:
        raise ValueError(f"Dataset inválido: {dataset}. Opções: {list(MODEL_REGISTRY.keys())}")
    if backbone not in MODEL_REGISTRY[dataset]["classical"]:
        raise ValueError(f"Backbone inválido: {backbone}.")
    if quantum_variant not in MODEL_REGISTRY[dataset]["quantum"]:
        raise ValueError(f"Quantum variant inválido: {quantum_variant}.")

    classical_model_path = MODEL_REGISTRY[dataset]["classical"][backbone]
    quantum_model_path = MODEL_REGISTRY[dataset]["quantum"][quantum_variant]
    output_dir = os.path.join(
        config["output_root"],
        dataset,
        f"classical_{backbone}__quantum_{quantum_variant}",
    )

    return classical_model_path, quantum_model_path, output_dir

CLASSICAL_MODEL_PATH, QUANTUM_MODEL_PATH, OUTPUT_DIR = _resolve_experiment_paths(EXPERIMENT_CONFIG)

NUM_CLASSES = EXPERIMENT_CONFIG["num_classes"]
IMAGE_SIZE = EXPERIMENT_CONFIG["image_size"]
BATCH_SIZE = EXPERIMENT_CONFIG["batch_size"]
N_QUBITS = EXPERIMENT_CONFIG["n_qubits"]
N_LAYERS = EXPERIMENT_CONFIG["n_layers"]
TARGET_FEATURE_DIM = EXPERIMENT_CONFIG["target_feature_dim"]

print("Configuração ativa:")
for k, v in EXPERIMENT_CONFIG.items():
    print(f"  - {k}: {v}")
print("\nCheckpoints resolvidos:")
print(f"  - Classical: {CLASSICAL_MODEL_PATH}")
print(f"  - Quantum:   {QUANTUM_MODEL_PATH}")
print(f"  - Output:    {OUTPUT_DIR}")

Configuração ativa:
  - dataset: INbreast
  - backbone: shallow
  - quantum_variant: basic
  - num_classes: 2
  - image_size: 28
  - batch_size: 8
  - n_qubits: 4
  - n_layers: 1
  - target_feature_dim: 128
  - output_root: experiments/features

Checkpoints resolvidos:
  - Classical: ../experiments/models/classical/INbreast/shallow/best_shallow.pth
  - Quantum:   ../experiments/models/quantum/basic/INbreast/checkpoints/model_epoch4.pth
  - Output:    experiments/features/INbreast/classical_shallow__quantum_basic


## 2. Carregar Modelos Pré-treinados

In [63]:
# Setup device
device = setup_device()
print(f"Using device: {device}")

# ============================================================================
# FUNÇÕES AUXILIARES PARA CRIAR MODELOS AUTOMATICAMENTE
# ============================================================================

def detect_quantum_model_type(checkpoint_path):
    """Detecta se o checkpoint é de QuantumModel ou ExtentedQuantumModel."""
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    
    if isinstance(checkpoint, dict):
        for key in ("model_state_dict", "state_dict", "model"):
            if key in checkpoint:
                checkpoint = checkpoint[key]
                break
    
    keys = set(checkpoint.keys())
    print(f"  ℹ️ Primeiras chaves do checkpoint: {list(keys)[:5]}")
    
    if any('quanv.' in k for k in keys):
        return "ExtentedQuantumModel"
    elif 'fc.weight' in keys and not any('quanv.' in k for k in keys):
        return "QuantumModel"
    else:
        print(f"⚠️ Não foi possível detectar o tipo. Chaves: {list(keys)[:10]}")
        return None

def build_classical_model(backbone, num_classes):
    if backbone == "resnet":
        return ClassicalModel(num_classes=num_classes)
    if backbone == "shallow":
        return ShallowCNN(num_classes=num_classes)
    raise ValueError(f"Backbone clássico não suportado: {backbone}")

def build_quantum_model(checkpoint_path, n_qubits, n_layers, num_classes):
    print("\nDetectando tipo de modelo quântico...")
    quantum_type = detect_quantum_model_type(checkpoint_path)
    print(f"  → Detectado: {quantum_type}")

    if quantum_type == "ExtentedQuantumModel":
        model = ExtentedQuantumModel(
            n_qubits=n_qubits,
            n_layers=n_layers,
            num_classes=num_classes,
            patch_size=2,
            mode="2d"
        )
        print("  ✓ Criado ExtentedQuantumModel (trainable)")
        return model

    if quantum_type == "QuantumModel":
        rand_params = np.random.uniform(low=0, high=2*np.pi, size=(n_layers, n_qubits))
        print(f"  ℹ️ Parâmetros aleatórios gerados: shape={rand_params.shape}")
        model = QuantumModel(
            n_qubits=n_qubits,
            n_layers=n_layers,
            rand_params=rand_params,
            num_classes=num_classes,
            patch_size=2
        )
        print("  ✓ Criado QuantumModel (non-trainable)")
        return model

    raise ValueError("❌ Não foi possível detectar o tipo de modelo quântico!")

def _load_model_checkpoint(model, checkpoint_path, device):
    """Carrega checkpoint de modelo, lidando com diferentes formatos."""
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

    if isinstance(checkpoint, dict):
        for key in ("model_state_dict", "state_dict", "model"):
            if key in checkpoint:
                checkpoint = checkpoint[key]
                break

    if isinstance(checkpoint, dict):
        checkpoint = {
            (k.replace("module.", "", 1) if k.startswith("module.") else k): v
            for k, v in checkpoint.items()
        }

    if hasattr(model, 'quanv'):
        print(f"  ℹ️ Inicializando modelo quântico trainable...")
        model.to(device)
        with torch.no_grad():
            dummy_input = torch.randn(1, 1, IMAGE_SIZE, IMAGE_SIZE).to(device)
            _ = model(dummy_input)

        print(f"  ℹ️ Carregando pesos do checkpoint (apenas FC)...")
        fc_state = {k: v for k, v in checkpoint.items() if k.startswith('fc.')}
        if fc_state:
            model.load_state_dict(fc_state, strict=False)
            print(f"  ✓ Carregados pesos FC: {list(fc_state.keys())}")
        else:
            print(f"  ⚠️ Nenhum peso FC encontrado no checkpoint")
        return model

    if hasattr(model, 'fc') and model.fc is None:
        print(f"  ℹ️ Inicializando camadas dinâmicas do {type(model).__name__}...")
        model.to(device)
        with torch.no_grad():
            dummy_input = torch.randn(1, 1, IMAGE_SIZE, IMAGE_SIZE).to(device)
            try:
                _ = model(dummy_input)
            except Exception as e:
                print(f"  ⚠️ Aviso ao inicializar: {e}")

    missing_keys, unexpected_keys = model.load_state_dict(checkpoint, strict=False)
    if missing_keys:
        print(f"  ⚠️ Chaves faltantes: {missing_keys}")
    if unexpected_keys:
        print(f"  ⚠️ Chaves inesperadas (ignoradas): {unexpected_keys}")

    return model

# ============================================================================
# CRIAR MODELOS A PARTIR DA CONFIGURAÇÃO ÚNICA
# ============================================================================

classical_model = build_classical_model(EXPERIMENT_CONFIG["backbone"], NUM_CLASSES)
quantum_model = build_quantum_model(QUANTUM_MODEL_PATH, N_QUBITS, N_LAYERS, NUM_CLASSES)

print("\nCarregando modelos...")
print("  → Carregando quantum model...")
_load_model_checkpoint(quantum_model, QUANTUM_MODEL_PATH, device)
print("  → Carregando classical model...")
_load_model_checkpoint(classical_model, CLASSICAL_MODEL_PATH, device)

quantum_model.to(device).eval()
classical_model.to(device).eval()

print("\n✓ Modelos carregados com sucesso!")
print(f"  - Quantum: {type(quantum_model).__name__}")
print(f"  - Classical: {type(classical_model).__name__}")

Using: NVIDIA GeForce RTX 4070 Ti SUPER
CUDA: 12.4
Using device: cuda

Detectando tipo de modelo quântico...
  ℹ️ Primeiras chaves do checkpoint: ['fc.bias', 'fc.weight']
  → Detectado: QuantumModel
  ℹ️ Parâmetros aleatórios gerados: shape=(1, 4)
  ✓ Criado QuantumModel (non-trainable)

Carregando modelos...
  → Carregando quantum model...
  ℹ️ Inicializando camadas dinâmicas do QuantumModel...
  → Carregando classical model...

✓ Modelos carregados com sucesso!
  - Quantum: QuantumModel
  - Classical: ShallowCNN


## 3. Criar Feature Extractors

## 3. Escolher Modo de Extração

Agora a escolha de extractor é **automática** via `create_extractor()` com base no tipo real do modelo carregado.

Você só precisa ajustar `EXPERIMENT_CONFIG` na seção de configurações:
- `dataset`
- `backbone`
- `quantum_variant`

O notebook faz a correspondência modelo → extractor sem mudanças manuais nesta etapa.

In [64]:
# ============================================================================
# EXTRACTOR AUTOMÁTICO A PARTIR DO TIPO DO MODELO (sem trocar manualmente)
# ============================================================================

def _resolve_extractor(model):
    # create_extractor já escolhe o extractor correto para cada arquitetura
    return create_extractor(model).to(device)

classical_extractor = _resolve_extractor(classical_model)
quantum_extractor = _resolve_extractor(quantum_model)

print("✓ Feature extractors criados automaticamente!")
print(f"  - Classical extractor: {type(classical_extractor).__name__}")
print(f"  - Quantum extractor: {type(quantum_extractor).__name__}")

✓ Feature extractors criados automaticamente!
  - Classical extractor: GenericFeatureExtractor
  - Quantum extractor: GenericFeatureExtractor


## 4. Preparar Dados

In [65]:
# Carregar dataloaders
loader = BreastMNISTLoader(batch_size=BATCH_SIZE, image_size=IMAGE_SIZE)
train_loader, test_loader, val_loader = loader.get_dataloaders()

splits = {
    "train": train_loader,
    "test": test_loader,
    "val": val_loader
}

print(f"✓ Dataloaders prontos:")
print(f"  - Train batches: {len(train_loader)}")
print(f"  - Test batches: {len(test_loader)}")
print(f"  - Val batches: {len(val_loader)}")

Using downloaded and verified file: /home/eflammere/.medmnist/breastmnist.npz
Using downloaded and verified file: /home/eflammere/.medmnist/breastmnist.npz
Using downloaded and verified file: /home/eflammere/.medmnist/breastmnist.npz
✓ Dataloaders prontos:
  - Train batches: 69
  - Test batches: 20
  - Val batches: 10


## 5. Função de Extração

Esta função extrai as features usando o extractor escolhido acima.

**Métodos de extração customizados:**
- Se quiser modificar o método de extração, edite a linha `features = extractor(imgs)`
- Exemplos de customização:
  - Extrair apenas da camada específica
  - Aplicar transformações adicionais
  - Combinar múltiplas camadas

In [66]:
def extract_features(extractor, dataloader, output_dir, split_name, target_dim=None):
    """Extrai features de um dataloader e salva.
    
    Args:
        extractor: Feature extractor
        dataloader: DataLoader
        output_dir: Diretório de saída
        split_name: Nome do split
        target_dim: Dimensão alvo (opcional). Se especificado, reduz features para essa dimensão
    """
    os.makedirs(output_dir, exist_ok=True)
    
    features_list = []
    labels_list = []
    
    extractor.eval()
    with torch.no_grad():
        for imgs, labels in tqdm(dataloader, desc=f"Extraindo {split_name}"):
            imgs = imgs.to(device)
            
            # ================================================================
            # MÉTODO DE EXTRAÇÃO - Modifique aqui se necessário
            # ================================================================
            
            # Método padrão: usar o extractor diretamente
            features = extractor(imgs)
            
            # Se target_dim especificado, reduzir dimensionalidade
            if target_dim is not None and features.shape[1] != target_dim:
                # Reduzir usando adaptive pooling 1D
                features = features.view(features.size(0), -1)
                features = torch.nn.functional.adaptive_avg_pool1d(
                    features.unsqueeze(1), 
                    target_dim
                ).squeeze(1)
            
            # Exemplos de métodos customizados (comente o padrão e descomente um destes):
            
            # Exemplo 1: Normalizar features após extração
            # features = extractor(imgs)
            # features = torch.nn.functional.normalize(features, p=2, dim=1)
            
            # Exemplo 2: Aplicar pooling adicional manual
            # features = extractor(imgs)
            # features = features.view(features.size(0), -1)
            # features = torch.nn.functional.adaptive_avg_pool1d(features.unsqueeze(1), 128).squeeze(1)
            
            # Exemplo 3: Extrair e concatenar múltiplas camadas (requer modificação do extractor)
            # features_layer1 = extractor.layer1(imgs)
            # features_layer2 = extractor.layer2(imgs)
            # features = torch.cat([features_layer1, features_layer2], dim=1)
            
            # ================================================================
            
            features_list.append(features.cpu().numpy())
            labels_list.append(labels.cpu().numpy())
    
    # Concatenar e salvar
    features = np.concatenate(features_list, axis=0)
    labels = np.concatenate(labels_list, axis=0)
    
    np.save(os.path.join(output_dir, f"{split_name}_features.npy"), features)
    np.save(os.path.join(output_dir, f"{split_name}_labels.npy"), labels)
    
    print(f"  ✓ Salvos: {split_name}_features.npy e {split_name}_labels.npy")
    
    return features.shape

## 6. Extrair Features

Execute a extração para todos os splits:

In [67]:
# Criar diretórios de saída
classical_dir = os.path.join(OUTPUT_DIR, "classical")
quantum_dir = os.path.join(OUTPUT_DIR, "quantum")

os.makedirs(classical_dir, exist_ok=True)
os.makedirs(quantum_dir, exist_ok=True)

print(f"Salvando features em: {OUTPUT_DIR}")
print(f"Target feature dim: {TARGET_FEATURE_DIM}\n")

# Extrair para cada split
for split_name, split_loader in splits.items():
    print(f"\n{'='*60}")
    print(f"Processando: {split_name.upper()}")
    print(f"{'='*60}")
    
    # Features clássicas
    print("\n[Classical]")
    c_shape = extract_features(classical_extractor, split_loader, classical_dir, 
                               split_name, target_dim=TARGET_FEATURE_DIM)
    print(f"  Shape: {c_shape}")
    
    # Features quânticas
    print("\n[Quantum]")
    q_shape = extract_features(quantum_extractor, split_loader, quantum_dir, 
                               split_name, target_dim=TARGET_FEATURE_DIM)
    print(f"  Shape: {q_shape}")

print(f"\n\n{'='*60}")
print("✓ EXTRAÇÃO COMPLETA!")
print(f"{'='*60}")
print(f"\nArquivos salvos em: {OUTPUT_DIR}")
if TARGET_FEATURE_DIM is not None:
    print(f"Dimensão das features: {TARGET_FEATURE_DIM}")

Salvando features em: experiments/features/INbreast/classical_shallow__quantum_basic
Target feature dim: 128


Processando: TRAIN

[Classical]


Extraindo train: 100%|██████████| 69/69 [00:00<00:00, 671.64it/s]


  ✓ Salvos: train_features.npy e train_labels.npy
  Shape: (546, 128)

[Quantum]


Extraindo train: 100%|██████████| 69/69 [01:45<00:00,  1.53s/it]


  ✓ Salvos: train_features.npy e train_labels.npy
  Shape: (546, 128)

Processando: TEST

[Classical]


...

  ✓ Salvos: test_features.npy e test_labels.npy
  Shape: (156, 128)

Processando: VAL

[Classical]


Extraindo val: 100%|██████████| 10/10 [00:00<00:00, 496.15it/s]


  ✓ Salvos: val_features.npy e val_labels.npy
  Shape: (78, 128)

[Quantum]


Extraindo val: 100%|██████████| 10/10 [00:15<00:00,  1.57s/it]

  ✓ Salvos: val_features.npy e val_labels.npy
  Shape: (78, 128)


✓ EXTRAÇÃO COMPLETA!

Arquivos salvos em: experiments/features/INbreast/classical_shallow__quantum_basic
Dimensão das features: 128


## 7. Verificar Features Extraídas

Verifique as dimensões e características das features extraídas:

In [68]:
# Carregar e verificar features de todos os splits
print("="*60)
print("RESUMO DAS FEATURES EXTRAÍDAS")
print("="*60)

for split_name in ["train", "val", "test"]:
    print(f"\n{split_name.upper()} Split:")
    
    c_features = np.load(os.path.join(classical_dir, f"{split_name}_features.npy"))
    q_features = np.load(os.path.join(quantum_dir, f"{split_name}_features.npy"))
    labels = np.load(os.path.join(classical_dir, f"{split_name}_labels.npy"))
    
    print(f"  Classical features: {c_features.shape}")
    print(f"  Quantum features:   {q_features.shape}")
    print(f"  Labels:             {labels.shape}")

print(f"\n{'='*60}")
print("Arquivos salvos em:")
print(f"  Classical: {classical_dir}/")
print(f"  Quantum:   {quantum_dir}/")
print("="*60)

RESUMO DAS FEATURES EXTRAÍDAS

TRAIN Split:
  Classical features: (546, 128)
  Quantum features:   (546, 128)
  Labels:             (546, 1)

VAL Split:
  Classical features: (78, 128)
  Quantum features:   (78, 128)
  Labels:             (78, 1)

TEST Split:
  Classical features: (156, 128)
  Quantum features:   (156, 128)
  Labels:             (156, 1)

Arquivos salvos em:
  Classical: experiments/features/INbreast/classical_shallow__quantum_basic/classical/
  Quantum:   experiments/features/INbreast/classical_shallow__quantum_basic/quantum/
